# PRIMA PROVA SVM

---

A quanto pare python può leggere i file `.mat`

In [1]:
from scipy import io as s_io
mat = s_io.loadmat('data/spectra.mat')

In [2]:
import numpy as np

ottani = np.zeros(mat['octane'].shape[0], dtype=np.float64)
spettri = np.zeros((mat['NIR'].shape[0], mat['NIR'].shape[1]), dtype=np.float64)
for i, (o, s) in enumerate(zip(mat['octane'], mat['NIR'])):
    ottani[i] = float(mat['octane'][i][0])
    spettri[i] = s

In [ ]:
# indici che ordinerebbero l'array ottano in modo crescente
indici_ordinamento = np.argsort(ottani)

# applico indici a entrambi per riordinarli in sincrono
ottano_ordinato = ottani[indici_ordinamento]
spettri_ordinati = spettri[indici_ordinamento]

---

## separazione in gruppos

le da dos o tres giros a cartoggi

In [5]:
N = ottano_ordinato.shape[0]

# vettore delle risposte
labels = np.concatenate((np.zeros(int(N/2)), np.ones(int(N/2))))

lo fa marquito

## SVM

In [6]:
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn import svm

import pickle as pkl
import pandas as pd
import os

In [ ]:
rkf = RepeatedStratifiedKFold(n_splits=6, n_repeats=5, random_state=42)

N_COMPONENTS_OPTIONS = [2, 5, None]
C_OPTIONS = [0.00001, 0.1, 1, 10]
KERNEL_OPTIONS = ["linear", "poly", "rbf"] 
GAMMA_OPTIONS = ['scale', 'auto', 0.01, 1]

# 1. Definizione Pipeline #
pipe = Pipeline([
    # Step 1: Scaling (# NOTE: va inserito uno scaler placeholder)
    ("scaling", StandardScaler()),       
    
    # Step 2: Riduzione dimensionalità (PCA)
    ("reduce_dim", PCA(random_state=42)),
    
    # Step 3: Classificatore
    ("classify", svm.SVC(random_state=42)) 
])

# 2. Definizione griglia dei parametri #
param_grid = [
{
    # Per provare diversi oggetti Scaler
    "scaling": [StandardScaler(), MinMaxScaler(), RobustScaler()],
    
    # Per provare parametri specifici di uno step (PCA)
    "reduce_dim__n_components": N_COMPONENTS_OPTIONS, 
    
    # Per provare parametri del classificatore
    "classify__C": C_OPTIONS,
    
    # Per provare diversi kernel
    "classify__kernel": KERNEL_OPTIONS,
    
    # Gamma dei kernel
    "classify__gamma": GAMMA_OPTIONS,
},
{
    # BLOCCO 2: Senza PCA
    # Per provare diversi oggetti Scaler
    "scaling": [StandardScaler(), MinMaxScaler(), RobustScaler()],
    
    # Per provare parametri specifici di uno step (PCA)
    "reduce_dim": ['passthrough'], 
    
    # Per provare parametri del classificatore
    "classify__C": C_OPTIONS,
    
    # Per provare diversi kernel
    "classify__kernel": KERNEL_OPTIONS,
    
    # Gamma dei kernel
    "classify__gamma": GAMMA_OPTIONS,
}]

# 3. Configurazione GridSearch #
grid = GridSearchCV(
    pipe, 
    param_grid=param_grid, 
    cv=rkf,
    n_jobs=-1, # «Number of jobs to run in parallel. -1 means using all processors»
    scoring={
    'score': 'accuracy',
    'sensitivity': 'recall'  # recall = sensitivity
    },
    refit='score', # «For multiple metric evaluation, needs to be a str denoting the
    # scorer to use to find the best parameters for refitting the estimator at the end»
    return_train_score=False
    )

# 4. Training e Validation (su Segnale B) #
grid.fit(spettri, labels)

# 5. Risultati #
print(f"La miglior configurazione: {grid.best_params_}")
print(f"Fornisce accuracy in validation: {grid.best_score_:.4f}")

# Conversione dei risultati in DataFrame
results_df = pd.DataFrame(grid.cv_results_)

La miglior configurazione: {'classify__C': 1, 'classify__gamma': 'scale', 'classify__kernel': 'linear', 'reduce_dim__n_components': None, 'scaling': RobustScaler()}
Fornisce accuracy in validation: 0.8967


/Users/zosojack/coding-with-qiskit/miniconda3/envs/ml-seda-env/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:490: FitFailedWarning: 
4320 fits failed out of a total of 17280.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
4320 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/zosojack/coding-with-qiskit/miniconda3/envs/ml-seda-env/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/zosojack/coding-with-qiskit/miniconda3/envs/ml-seda-env/lib/python3.12/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^

In [8]:
# Ciascuna combinazione di parametri è una riga
print(f"Numero totale di configurazioni provate: {results_df.shape[0]}")

# Selezioniamo solo le colonne interessanti per pulire la vista
columns_to_show = [
    'param_reduce_dim__n_components',
    'param_scaling', 
    'param_classify__C', 
    'param_classify__kernel',
    'param_classify__gamma',
    'mean_test_score', 
    'std_test_score', 
    'mean_test_sensitivity',
    'rank_test_score'
]

# Ordiniamo per classifica (rank_test_score)
analysis = results_df[columns_to_show].sort_values('rank_test_score')

# Se si ha, è comodo aprire analysis in un viewer tipo Data Wrangler
analysis.head(20)

Numero totale di configurazioni provate: 576


,param_reduce_dim__n_components,param_scaling,param_classify__C,param_classify__kernel,param_classify__gamma,mean_test_score,std_test_score,mean_test_sensitivity,rank_test_score
224,None,RobustScaler(),1.0,linear,scale,0.896667,0.104828,0.906667,1
251,None,RobustScaler(),1.0,linear,auto,0.896667,0.104828,0.906667,1
305,None,RobustScaler(),1.0,linear,1,0.896667,0.104828,0.906667,1
278,None,RobustScaler(),1.0,linear,0.01,0.896667,0.104828,0.906667,1
276,None,StandardScaler(),1.0,linear,0.01,0.890000,0.097809,0.906667,5
249,None,StandardScaler(),1.0,linear,auto,0.890000,0.097809,0.906667,5
222,None,StandardScaler(),1.0,linear,scale,0.890000,0.097809,0.906667,5
303,None,StandardScaler(),1.0,linear,1,0.890000,0.097809,0.906667,5
347,5,RobustScaler(),10.0,rbf,scale,0.876667,0.098939,0.860000,9
359,None,RobustScaler(),10.0,linear,auto,0.866667,0.104350,0.886667,10


In [ ]:
results_df.to_pickle("results/results_svm_GridSearch.pkl")